# Music Discovery Engine — ALS Embedding Analysis

Showcases the benefit of the ALS collaborative-filtering embedding approach over naive popularity ranking.

**Pipeline**: Raw play counts → ALS (128-dim) → GMM personas → diversity-aware reranker

**Sections**
1. Data overview
2. The popularity-bias problem
3. ALS embedding quality
4. Song embedding space (UMAP)
5. User embedding space (UMAP)
6. Persona discovery (case study)
7. System performance vs baselines
8. Summary

In [1]:
import sys
sys.path.insert(0, '..')

import warnings; warnings.filterwarnings('ignore')
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA

DATA_DIR       = Path('../data/processed')
EMBEDDINGS_DIR = Path('../models/embeddings')
PERSONAS_DIR   = Path('../models/personas')
RESULTS_DIR    = Path('../results/holdout')

plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Setup done.')

Setup done.


## 1. Data Overview

In [2]:
interactions = pd.read_parquet(DATA_DIR / 'interactions.parquet')
songs_df     = pd.read_parquet(DATA_DIR / 'songs.parquet')

n_users = interactions['user_id'].nunique()
n_songs = interactions['song_id'].nunique()
splits  = interactions['split'].value_counts()

print(f'Total interactions : {len(interactions):,}')
print(f'Unique users       : {n_users:,}')
print(f'Unique songs       : {n_songs:,}')
print(f'Split breakdown    : {splits.to_dict()}')
print(f'Avg plays / user   : {interactions.groupby("user_id")["play_count"].sum().mean():.1f}')
print(f'Avg songs / user   : {interactions.groupby("user_id")["song_id"].count().mean():.1f}')
interactions.head()

Total interactions : 91,331
Unique users       : 2,000
Unique songs       : 41,278
Split breakdown    : {'train': 67091, 'test': 12120, 'val': 12120}
Avg plays / user   : 129.1
Avg songs / user   : 45.7


,user_id,song_id,play_count,weight,split
0,83a05753542608b67700a6655a91aa10a0372568,SOILFUU12AB017C75F,17,2.890372,test
1,83a05753542608b67700a6655a91aa10a0372568,SOAXGDH12A8C13F8A1,37,3.637586,test
2,83a05753542608b67700a6655a91aa10a0372568,SOEGECP12B0B80B3FD,1,0.693147,test
3,83a05753542608b67700a6655a91aa10a0372568,SOUCPBK12A58A7881A,5,1.791759,test
4,83a05753542608b67700a6655a91aa10a0372568,SOGPBDO12A6D4F7F85,1,0.693147,train


## 2. The Problem: Popularity Bias

Popularity-based recommenders dominate with a tiny fraction of songs. This notebook quantifies that gap and shows how ALS + personas fix it.

In [3]:
train = interactions[interactions['split'] == 'train']
song_plays = train.groupby('song_id')['play_count'].sum().sort_values(ascending=False)
total_plays = song_plays.sum()
cumsum = song_plays.cumsum() / total_plays

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Lorenz curve
ax = axes[0]
pct_songs = np.linspace(0, 1, len(song_plays))
ax.plot(pct_songs * 100, cumsum.values * 100, color='#4C72B0', lw=2)
ax.plot([0, 100], [0, 100], 'k--', alpha=0.35, label='Perfect equality')
top1_idx = max(1, int(0.01 * len(song_plays)))
top1_share = float(cumsum.iloc[top1_idx]) * 100
ax.fill_between(pct_songs[:top1_idx+1] * 100, cumsum.values[:top1_idx+1] * 100,
                alpha=0.2, color='#4C72B0')
ax.annotate(f'Top 1% songs\n→ {top1_share:.0f}% of plays',
            xy=(1, top1_share), xytext=(14, top1_share - 18), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_xlabel('% of songs (ranked by popularity)')
ax.set_ylabel('% of total plays')
ax.set_title('Popularity Concentration (Lorenz Curve)')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

# Play-count histogram (log)
ax = axes[1]
plays = song_plays.values
ax.hist(np.log10(plays + 1), bins=60, color='#DD8452', edgecolor='none', alpha=0.85)
ax.axvline(np.log10(plays.mean() + 1), color='red',   lw=1.5, ls='--',
           label=f'Mean = {plays.mean():.0f}')
ax.axvline(np.log10(np.median(plays) + 1), color='green', lw=1.5, ls='--',
           label=f'Median = {int(np.median(plays))}')
ax.set_xlabel('log₁₀(play count + 1)')
ax.set_ylabel('Number of songs')
ax.set_title('Song Play Count Distribution')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

fig.suptitle('Why popularity ranking fails: the long-tail problem', fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(EMBEDDINGS_DIR.parent / 'popularity_bias.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Top 1% of songs account for {top1_share:.1f}% of all training plays.')
print(f'A popularity ranker ignores {100-top1_share:.1f}% of the listening signal.')

Top 1% of songs account for 21.2% of all training plays.
A popularity ranker ignores 78.8% of the listening signal.


## 3. ALS Embedding Quality

ALS learns 128-dim latent factors purely from co-listening patterns — no hand-crafted features.

In [4]:
song_emb_df = pd.read_parquet(EMBEDDINGS_DIR / 'song_embeddings.parquet')
user_emb_df = pd.read_parquet(EMBEDDINGS_DIR / 'user_embeddings.parquet')

emb_cols   = [c for c in song_emb_df.columns if c.startswith('emb_')]
usrc_cols  = [c for c in user_emb_df.columns if c.startswith('emb_')]

song_emb_mat = song_emb_df[emb_cols].to_numpy(dtype=np.float32)
user_emb_mat = user_emb_df[usrc_cols].to_numpy(dtype=np.float32)

# Cosine-normalise for similarity comparisons
song_norms  = np.linalg.norm(song_emb_mat, axis=1, keepdims=True)
user_norms  = np.linalg.norm(user_emb_mat, axis=1, keepdims=True)
song_emb_n  = song_emb_mat / np.where(song_norms > 0, song_norms, 1.0)
user_emb_n  = user_emb_mat / np.where(user_norms > 0, user_norms, 1.0)

song_to_pos = {sid: i for i, sid in enumerate(song_emb_df['song_id'])}
user_to_pos = {uid: i for i, uid in enumerate(user_emb_df['user_id'])}

# Merge artist info
song_emb_df = song_emb_df.merge(songs_df[['song_id', 'artist_name']], on='song_id', how='left')

print(f'Song embeddings : {song_emb_mat.shape}  |  catalog coverage: {len(song_emb_df)}/{n_songs} songs ({100*len(song_emb_df)/n_songs:.1f}%)')
print(f'User embeddings : {user_emb_mat.shape}')
print(f'Embedding dim   : {len(emb_cols)}')
print(f'Norm mean±std   : {song_norms.mean():.4f} ± {song_norms.std():.4f}  (ALS vectors, not unit-normalised)')

Song embeddings : (33959, 128)  |  catalog coverage: 33959/41278 songs (82.3%)
User embeddings : (2000, 128)
Embedding dim   : 128
Norm mean±std   : 0.1761 ± 0.1635  (ALS vectors, not unit-normalised)


In [5]:
# Test: are same-artist songs closer in embedding space than random pairs?
rng = np.random.default_rng(42)

artist_to_positions = {}
for artist, grp in song_emb_df.groupby('artist_name'):
    positions = [song_to_pos[sid] for sid in grp['song_id'] if sid in song_to_pos]
    if len(positions) >= 2:
        artist_to_positions[artist] = positions

N = 4000
artists_with_pairs = list(artist_to_positions.keys())

same_sims, diff_sims, rand_sims = [], [], []
for _ in range(N):
    # same artist
    a = rng.choice(artists_with_pairs)
    i, j = rng.choice(artist_to_positions[a], size=2, replace=False)
    same_sims.append(float(song_emb_n[i] @ song_emb_n[j]))
    # different artist
    a1, a2 = rng.choice(artists_with_pairs, size=2, replace=False)
    pi = rng.choice(artist_to_positions[a1])
    pj = rng.choice(artist_to_positions[a2])
    diff_sims.append(float(song_emb_n[pi] @ song_emb_n[pj]))
    # random
    ri, rj = rng.integers(0, len(song_emb_n), 2)
    rand_sims.append(float(song_emb_n[ri] @ song_emb_n[rj]))

same_sims = np.array(same_sims)
diff_sims = np.array(diff_sims)
rand_sims  = np.array(rand_sims)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Histogram comparison
ax = axes[0]
bins = np.linspace(-0.6, 1.0, 80)
ax.hist(rand_sims,  bins=bins, alpha=0.50, color='gray',     label=f'Random pairs     μ={rand_sims.mean():.3f}')
ax.hist(diff_sims,  bins=bins, alpha=0.55, color='#DD8452',  label=f'Diff-artist pairs μ={diff_sims.mean():.3f}')
ax.hist(same_sims,  bins=bins, alpha=0.65, color='#4C72B0',  label=f'Same-artist pairs μ={same_sims.mean():.3f}')
ax.set_xlabel('Cosine similarity')
ax.set_ylabel('Count')
ax.set_title('ALS Embeds Artist Cohesion\nwithout explicit artist labels')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

# Mean + 95% CI bar chart
ax = axes[1]
groups   = ['Random', 'Diff artist', 'Same artist']
arr_list = [rand_sims, diff_sims, same_sims]
colors   = ['gray', '#DD8452', '#4C72B0']
means    = [a.mean() for a in arr_list]
ci95     = [1.96 * a.std() / np.sqrt(len(a)) for a in arr_list]
ax.bar(groups, means, color=colors, alpha=0.85, yerr=ci95, capsize=5, edgecolor='white')
ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.4)
ax.set_ylabel('Mean cosine similarity (95% CI)')
ax.set_title('Embedding Proximity by Pair Type')
ax.grid(axis='y', alpha=0.2)

fig.suptitle('Collaborative signal: ALS captures artist-level taste without hand-crafted features',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(EMBEDDINGS_DIR / 'embedding_artist_similarity.png', bbox_inches='tight', dpi=150)
plt.show()

from scipy import stats
t, p = stats.ttest_ind(same_sims, rand_sims)
print(f'Same-artist vs random: Δμ = {same_sims.mean()-rand_sims.mean():.4f},  t={t:.1f},  p={p:.2e}')

Same-artist vs random: Δμ = 0.2791,  t=40.8,  p=0.00e+00


## 4. Song Embedding Space (UMAP)

Each point = one song. Colour = artist (top 12 by play count). Structure emerges purely from co-listening.

In [6]:
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print('umap-learn not installed — using PCA. Run: pip install umap-learn')

N_VIZ = 5000
rng42 = np.random.default_rng(42)
viz_idx = rng42.choice(len(song_emb_n), size=min(N_VIZ, len(song_emb_n)), replace=False)
viz_emb = song_emb_n[viz_idx]
viz_artists = song_emb_df['artist_name'].iloc[viz_idx].fillna('Unknown').to_numpy()

if HAS_UMAP:
    reducer = umap.UMAP(n_neighbors=25, min_dist=0.08, random_state=42, n_jobs=1)
    coords  = reducer.fit_transform(viz_emb)
    method  = 'UMAP'
else:
    pca    = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(viz_emb)
    method = 'PCA'

# Top 12 artists by training play count
top_artists = (
    train.merge(songs_df[['song_id','artist_name']], on='song_id', how='left')
    .groupby('artist_name')['play_count'].sum()
    .sort_values(ascending=False)
    .head(12).index.tolist()
)

cmap12 = plt.cm.get_cmap('tab20', len(top_artists))
artist_color = {a: cmap12(i) for i, a in enumerate(top_artists)}

fig, ax = plt.subplots(figsize=(11, 8))

# Background: non-top songs in light gray
mask_other = ~np.isin(viz_artists, top_artists)
ax.scatter(coords[mask_other, 0], coords[mask_other, 1],
           s=3, alpha=0.18, color='lightgray', rasterized=True)

# Foreground: top artists
for artist in top_artists:
    mask = viz_artists == artist
    if mask.sum() == 0:
        continue
    ax.scatter(coords[mask, 0], coords[mask, 1],
               s=14, alpha=0.75, color=artist_color[artist],
               label=artist, rasterized=True)

ax.set_title(f'{method} of ALS Song Embeddings\n'
             f'{N_VIZ:,} songs — coloured by top-12 artists (gray = all others)',
             fontsize=12)
ax.set_xlabel(f'{method}-1')
ax.set_ylabel(f'{method}-2')
ax.legend(markerscale=2.5, fontsize=7.5, bbox_to_anchor=(1.01, 1), loc='upper left',
          frameon=False)
fig.tight_layout()
fig.savefig(EMBEDDINGS_DIR / 'umap_songs.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'2D projection via {method}. {N_VIZ:,} songs visualised.')

2D projection via UMAP. 5,000 songs visualised.


## 5. User Embedding Space

Each point = one user. Colour = number of unique songs in their training history (engagement level).

In [7]:
if HAS_UMAP:
    user_reducer = umap.UMAP(n_neighbors=20, min_dist=0.1, random_state=42, n_jobs=1)
    user_coords  = user_reducer.fit_transform(user_emb_n)
    umeth = 'UMAP'
else:
    pca_u        = PCA(n_components=2, random_state=42)
    user_coords  = pca_u.fit_transform(user_emb_n)
    umeth = 'PCA'

user_history_size = (
    train.groupby('user_id')['song_id'].nunique()
    .reindex(user_emb_df['user_id']).fillna(0).to_numpy()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Colour by history size
ax = axes[0]
sc = ax.scatter(user_coords[:, 0], user_coords[:, 1],
                c=np.log1p(user_history_size),
                cmap='plasma', s=6, alpha=0.6, rasterized=True)
plt.colorbar(sc, ax=ax, label='log(unique songs in history + 1)')
ax.set_title(f'{umeth} of User Embeddings\nColoured by listening breadth')
ax.set_xlabel(f'{umeth}-1'); ax.set_ylabel(f'{umeth}-2')

# Colour by total play count
user_play_total = (
    train.groupby('user_id')['play_count'].sum()
    .reindex(user_emb_df['user_id']).fillna(0).to_numpy()
)
ax = axes[1]
sc2 = ax.scatter(user_coords[:, 0], user_coords[:, 1],
                 c=np.log1p(user_play_total),
                 cmap='viridis', s=6, alpha=0.6, rasterized=True)
plt.colorbar(sc2, ax=ax, label='log(total play count + 1)')
ax.set_title(f'{umeth} of User Embeddings\nColoured by total play volume')
ax.set_xlabel(f'{umeth}-1'); ax.set_ylabel(f'{umeth}-2')

fig.suptitle('User latent space — engagement gradient visible across dimensions',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(EMBEDDINGS_DIR / 'umap_users.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Persona Discovery — Case Study

A single user's training songs projected to 2D and coloured by GMM persona assignment. Multiple distinct clusters = the user has different taste modes the system can serve.

In [8]:
# Find a user with k >= 3 personas and enough songs
best_user, best_persona = None, None
for persona_dir in sorted(PERSONAS_DIR.iterdir()):
    pkl = persona_dir / 'persona.pkl'
    if not pkl.exists():
        continue
    uid = persona_dir.name
    if uid not in user_to_pos:
        continue
    with open(pkl, 'rb') as f:
        p = pickle.load(f)
    if p.k >= 3:
        best_user, best_persona = uid, p
        break

if best_user is None:
    print('No user with k>=3 personas found — lower threshold to k>=2')
    for persona_dir in sorted(PERSONAS_DIR.iterdir()):
        pkl = persona_dir / 'persona.pkl'
        if not pkl.exists(): continue
        uid = persona_dir.name
        if uid not in user_to_pos: continue
        with open(pkl, 'rb') as f:
            p = pickle.load(f)
        if p.k >= 2:
            best_user, best_persona = uid, p
            break

if best_user is None:
    print('No valid persona found — run: music-discovery profile')
else:
    user_train_songs = set(train[train['user_id'] == best_user]['song_id'])
    hist_idx = np.array([song_to_pos[s] for s in user_train_songs if s in song_to_pos], dtype=np.intp)
    hist_emb  = song_emb_n[hist_idx]       # (N_hist, 128)
    assignments = best_persona.dominant_persona(hist_emb)  # persona index per song

    if HAS_UMAP and len(hist_emb) >= 15:
        h_reducer = umap.UMAP(n_neighbors=min(15, len(hist_emb)-1), min_dist=0.08,
                               random_state=42, n_jobs=1)
        h2d = h_reducer.fit_transform(hist_emb)
        h_method = 'UMAP'
    else:
        h2d = PCA(n_components=2, random_state=42).fit_transform(hist_emb)
        h_method = 'PCA'

    persona_colors = plt.cm.get_cmap('Set1', best_persona.k)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    # Left: songs coloured by persona
    ax = axes[0]
    for ki in range(best_persona.k):
        mask = assignments == ki
        sids_in_persona = [s for s, m in zip(user_train_songs, [assignments[j]==ki for j in range(len(assignments))]) if m]
        ax.scatter(h2d[mask, 0], h2d[mask, 1],
                   s=20, alpha=0.8, color=persona_colors(ki),
                   label=f'Persona {ki+1}  (n={mask.sum()})')
    ax.set_title(f'{h_method} of training songs\nUser: {best_user[:12]}...')
    ax.set_xlabel(f'{h_method}-1'); ax.set_ylabel(f'{h_method}-2')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)

    # Right: persona weight bar + size bar
    ax = axes[1]
    persona_labels = [f'Persona {i+1}' for i in range(best_persona.k)]
    weights = best_persona.weights
    counts  = [(assignments == i).sum() for i in range(best_persona.k)]
    x = np.arange(best_persona.k)
    width = 0.35
    b1 = ax.bar(x - width/2, weights, width, label='GMM weight', color=[persona_colors(i) for i in range(best_persona.k)], alpha=0.85)
    ax2 = ax.twinx()
    b2 = ax2.bar(x + width/2, counts, width, label='Song count', color=[persona_colors(i) for i in range(best_persona.k)], alpha=0.45)
    ax.set_xticks(x); ax.set_xticklabels(persona_labels)
    ax.set_ylabel('GMM mixture weight')
    ax2.set_ylabel('Songs in persona')
    ax.set_title(f'Persona breakdown  (k={best_persona.k} components)')
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

    fig.suptitle('GMM personas reveal distinct taste modes in a single user\'s history',
                 fontsize=12, fontweight='bold')
    fig.tight_layout()
    fig.savefig(EMBEDDINGS_DIR.parent / 'persona_case_study.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'User {best_user[:16]}...  k={best_persona.k} personas,  {len(hist_idx)} training songs')
    for ki in range(best_persona.k):
        n = (assignments == ki).sum()
        print(f'  Persona {ki+1}: weight={best_persona.weights[ki]:.3f}, songs={n}')

User 001c75b8dc6a6712...  k=8 personas,  41 training songs
  Persona 1: weight=0.087, songs=9
  Persona 2: weight=0.652, songs=32
  Persona 3: weight=0.109, songs=0
  Persona 4: weight=0.022, songs=0
  Persona 5: weight=0.022, songs=0
  Persona 6: weight=0.043, songs=0
  Persona 7: weight=0.043, songs=0
  Persona 8: weight=0.022, songs=0


## 7. System Performance vs Baselines

Compares three models on the held-out test split:
- **Popularity** — recommend most-played songs globally
- **ALS** — recommend by dot product of user × song embeddings
- **Persona reranker** — ALS shortlist reranked by GMM persona scorer

Run `music-discovery evaluate holdout` to generate results.

In [9]:
summary_path  = RESULTS_DIR / 'holdout_summary.csv'
metrics_path  = RESULTS_DIR / 'holdout_metrics.csv'

if not summary_path.exists():
    print('No holdout results found.')
    print('Run:  music-discovery evaluate holdout')
else:
    summary = pd.read_csv(summary_path, index_col=0)
    metrics = pd.read_csv(metrics_path)
    print('Holdout summary (mean over users):')
    display_cols = [c for c in summary.columns if any(k in c for k in ['Recall','NDCG','HitRate','ILD','long_tail','fallback','coverage'])]
    print(summary[display_cols].round(4).to_string())

    model_order  = ['popularity', 'als', 'persona_reranker']
    model_labels = {'popularity': 'Popularity', 'als': 'ALS', 'persona_reranker': 'Persona reranker'}
    colors       = {'popularity': '#C44E52', 'als': '#4C72B0', 'persona_reranker': '#55A868'}
    available    = [m for m in model_order if m in summary.index]

    k_val = 10
    raw_pairs = [(f'Recall@{k_val}', f'Recall_adj@{k_val}', 'Recall'),
                 (f'NDCG@{k_val}',   f'NDCG_adj@{k_val}',   'NDCG')]
    pair_ok = [(r,a,l) for r,a,l in raw_pairs if r in summary.columns and a in summary.columns]

    n_plots = len(pair_ok) + 2  # +2 for HitRate and ILD
    fig, axes = plt.subplots(1, n_plots, figsize=(5.5 * n_plots, 5))
    if n_plots == 1: axes = [axes]

    ax_idx = 0
    # Raw vs Adjusted Recall / NDCG
    for raw_col, adj_col, label in pair_ok:
        ax = axes[ax_idx]; ax_idx += 1
        x = np.arange(len(available)); w = 0.35
        raw_vals = [summary.loc[m, raw_col] for m in available]
        adj_vals = [summary.loc[m, adj_col] for m in available]
        ax.bar(x - w/2, raw_vals, w, label='Raw',      color=[colors[m] for m in available], alpha=0.55)
        ax.bar(x + w/2, adj_vals, w, label='Adj',      color=[colors[m] for m in available], alpha=0.95)
        ax.set_xticks(x)
        ax.set_xticklabels([model_labels[m] for m in available], rotation=15, ha='right')
        ax.set_title(f'{label}@{k_val}\nraw (pale) vs coverage-adjusted')
        ax.set_ylim(0, max(max(raw_vals+adj_vals)*1.2, 0.01))
        ax.grid(axis='y', alpha=0.2)
        if ax_idx == 1: ax.legend(fontsize=9)

    # HitRate
    if f'HitRate@{k_val}' in summary.columns:
        ax = axes[ax_idx]; ax_idx += 1
        vals = [summary.loc[m, f'HitRate@{k_val}'] for m in available]
        ax.bar([model_labels[m] for m in available], vals,
               color=[colors[m] for m in available], alpha=0.88)
        ax.set_title(f'HitRate@{k_val}')
        ax.set_ylim(0, 1); ax.grid(axis='y', alpha=0.2)
        for i, v in enumerate(vals):
            ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

    # ILD
    if 'ILD' in summary.columns:
        ax = axes[ax_idx]; ax_idx += 1
        vals = [summary.loc[m, 'ILD'] for m in available]
        ax.bar([model_labels[m] for m in available], vals,
               color=[colors[m] for m in available], alpha=0.88)
        ax.set_title('Intra-List Diversity (ILD)\nhigher = more diverse recs')
        ax.grid(axis='y', alpha=0.2)
        for i, v in enumerate(vals):
            ax.text(i, v + 0.001, f'{v:.3f}', ha='center', fontsize=9)

    fig.suptitle('Holdout evaluation — test split (val used for weight optimisation only)',
                 fontsize=12, fontweight='bold')
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'holdout_comparison.png', bbox_inches='tight', dpi=150)
    plt.show()

No holdout results found.
Run:  music-discovery evaluate holdout


## 8. Diversity–Relevance Trade-off

Each point = one user. The best model sits top-right: high ILD (diverse list) AND high Recall.

In [10]:
if not metrics_path.exists():
    print('Run music-discovery evaluate holdout first.')
else:
    k_val = 10
    fig, ax = plt.subplots(figsize=(8, 6))

    for model in available:
        sub = metrics[metrics['model'] == model]
        recall_col = f'Recall_adj@{k_val}' if f'Recall_adj@{k_val}' in sub.columns else f'Recall@{k_val}'
        x = sub['ILD'].to_numpy()
        y = sub[recall_col].to_numpy()
        ax.scatter(x, y, s=12, alpha=0.4, color=colors[model], label=model_labels[model], rasterized=True)
        ax.scatter(x.mean(), y.mean(), s=120, color=colors[model],
                   marker='D', edgecolors='white', linewidths=1.2, zorder=5)
        ax.annotate(f'{model_labels[model]}\nμ=({x.mean():.3f}, {y.mean():.3f})',
                    xy=(x.mean(), y.mean()), xytext=(8, 4), textcoords='offset points',
                    fontsize=8, color=colors[model])

    ax.set_xlabel('Intra-List Diversity (ILD)')
    ax.set_ylabel(f'{"Recall_adj" if f"Recall_adj@{k_val}" in metrics.columns else "Recall"}@{k_val}')
    ax.set_title('Diversity–Relevance Trade-off per User\n(◆ = per-model mean)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'diversity_relevance_scatter.png', bbox_inches='tight', dpi=150)
    plt.show()

Run music-discovery evaluate holdout first.


## 9. Summary

In [11]:
print('=' * 58)
print('MUSIC DISCOVERY ENGINE — ANALYSIS SUMMARY')
print('=' * 58)
print(f'  Dataset          : {n_users:,} users, {n_songs:,} songs')
print(f'  ALS embedding dim: {len(emb_cols)}')
print(f'  Catalog coverage : {len(song_emb_df):,} / {n_songs:,} songs ({100*len(song_emb_df)/n_songs:.1f}%)')
print(f'  Persona models   : {sum(1 for d in PERSONAS_DIR.iterdir() if (d/"persona.pkl").exists()):,} users')
print()
print(f'  Top 1% songs capture {top1_share:.1f}% of training plays (long-tail problem)')
print(f'  Same-artist cosine sim: {same_sims.mean():.4f}  vs  random: {rand_sims.mean():.4f}')
print()
if summary_path.exists():
    for model in available:
        r_col = f'Recall_adj@10' if 'Recall_adj@10' in summary.columns else 'Recall@10'
        n_col = f'NDCG_adj@10'   if 'NDCG_adj@10'   in summary.columns else 'NDCG@10'
        r = summary.loc[model, r_col] if r_col in summary.columns else float('nan')
        n = summary.loc[model, n_col] if n_col in summary.columns else float('nan')
        i = summary.loc[model, 'ILD'] if 'ILD' in summary.columns else float('nan')
        print(f'  {model_labels.get(model, model):20s}  Recall@10={r:.4f}  NDCG@10={n:.4f}  ILD={i:.4f}')
else:
    print('  Holdout results: not yet generated.')
    print('  Run: music-discovery evaluate holdout')
print()
print('Saved plots:')
for p in [
    EMBEDDINGS_DIR.parent / 'popularity_bias.png',
    EMBEDDINGS_DIR / 'embedding_artist_similarity.png',
    EMBEDDINGS_DIR / 'umap_songs.png',
    EMBEDDINGS_DIR / 'umap_users.png',
    EMBEDDINGS_DIR.parent / 'persona_case_study.png',
]:
    print(f"  {'✓' if p.exists() else '○'}  {p}")

MUSIC DISCOVERY ENGINE — ANALYSIS SUMMARY
  Dataset          : 2,000 users, 41,278 songs
  ALS embedding dim: 128
  Catalog coverage : 33,959 / 41,278 songs (82.3%)
  Persona models   : 6,259 users

  Top 1% songs capture 21.2% of training plays (long-tail problem)
  Same-artist cosine sim: 0.3262  vs  random: 0.0471

  Holdout results: not yet generated.
  Run: music-discovery evaluate holdout

Saved plots:
  ✓  ../models/popularity_bias.png
  ✓  ../models/embeddings/embedding_artist_similarity.png
  ✓  ../models/embeddings/umap_songs.png
  ✓  ../models/embeddings/umap_users.png
  ✓  ../models/persona_case_study.png
